# 📖 Notebook 2: Swipe and Match System

The core interaction on Tinder is simple: **swipe right** (like) or **swipe left** (pass). When two users both swipe right on each other, it's a **match**.

But here's the challenge: with 20M daily active users doing ~100 swipes each, that's **2 billion swipes per day**. And we need **strong consistency** — if User A and User B both swipe right at nearly the same time, the system MUST detect the match. Missing a match means two people who liked each other never connect.

This notebook shows:
1. Why a naive check-then-write approach has a **race condition**
2. How to solve it with **Redis Lua scripts** for atomic swipe + match detection
3. How the **user-pair key** pattern ensures both swipe directions hit the same Redis key

## Learning Objectives

By the end of this notebook, you'll understand:
- Why concurrent swipes create race conditions
- How Redis Lua scripts provide atomicity (all-or-nothing execution)
- The user-pair key pattern for partitioning swipe data
- How to persist matches durably to PostgreSQL after Redis detects them

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/tinder
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `tinder_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tinder_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def query(sql, params=None):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    """Execute a write query (INSERT/UPDATE/DELETE)."""
    conn = get_db()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.close()

# Test both connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Problem: Race Conditions in Swiping

Imagine two users swipe right on each other at almost the same time:

```
Timeline:
  T1: User A swipes right on User B → check DB for B→A swipe → NOT FOUND
  T2: User B swipes right on User A → check DB for A→B swipe → NOT FOUND
  T3: Save A→B swipe to DB
  T4: Save B→A swipe to DB
```

Both swipes are saved, but **neither detected the match**! The check happened before the other swipe was written. This is a classic **race condition** — the result depends on the timing of two concurrent operations.

Let's demonstrate this with code:

In [ ]:
# The NAIVE approach: separate check and write
# This has a race condition!

def naive_swipe(swiper_id: int, target_id: int, direction: str) -> bool:
    """Record a swipe and check for match. BUGGY — has race condition!"""
    conn = get_db()
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Step 1: Check if the other person already swiped right on us
    cursor.execute(
        "SELECT direction FROM swipes WHERE swiper_id = %s AND target_id = %s",
        (target_id, swiper_id)
    )
    other_swipe = cursor.fetchone()
    
    # ⚠️ GAP: Between this check and the write below, the other user could swipe!
    time.sleep(0.1)  # Simulate network/processing delay that widens the race window
    
    # Step 2: Save our swipe
    cursor.execute(
        "INSERT INTO swipes (swiper_id, target_id, direction) VALUES (%s, %s, %s) ON CONFLICT DO NOTHING",
        (swiper_id, target_id, direction)
    )
    
    conn.close()
    
    # Step 3: Check for match
    is_match = (direction == 'right' 
                and other_swipe is not None 
                and other_swipe[0] == 'right')
    return is_match

print("⚠️  The naive approach: check → (gap) → write")
print("   During the gap, the other user's swipe might not exist yet.")
print("   This means mutual swipes can MISS each other!")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Demonstrate the race condition with concurrent swipes
# We'll use users Mia (4) and Noah (14) — Mia already swiped right,
# but let's reset and simulate both swiping at the same time

# Clean up existing swipes between users 6 and 16 (Luna and Oliver)
execute("DELETE FROM swipes WHERE (swiper_id = 6 AND target_id = 16) OR (swiper_id = 16 AND target_id = 6)")

results = {}

def swipe_as(swiper, target, label):
    """Wrapper to capture the result."""
    match = naive_swipe(swiper, target, 'right')
    results[label] = match

# Both swipe right at the same time
print("🏁 Simulating concurrent swipes: Luna(6) ↔ Oliver(16)\n")

with ThreadPoolExecutor(max_workers=2) as pool:
    pool.submit(swipe_as, 6, 16, 'Luna→Oliver')
    pool.submit(swipe_as, 16, 6, 'Oliver→Luna')

time.sleep(0.5)  # Wait for threads to finish

print(f"   Luna→Oliver detected match?  {results.get('Luna→Oliver', '?')}")
print(f"   Oliver→Luna detected match?  {results.get('Oliver→Luna', '?')}")
print()

# Check: both swipes exist, but did anyone detect the match?
swipes = query("""
    SELECT swiper_id, target_id, direction 
    FROM swipes 
    WHERE (swiper_id = 6 AND target_id = 16) OR (swiper_id = 16 AND target_id = 6)
""")
print(f"📊 Swipes in DB: {len(swipes)}")
for s in swipes:
    print(f"   User {s['swiper_id']} → User {s['target_id']}: {s['direction']}")

if not results.get('Luna→Oliver') and not results.get('Oliver→Luna'):
    print()
    print("❌ RACE CONDITION! Both swiped right but NEITHER detected the match!")
    print("   Luna and Oliver will never know they liked each other 💔")
else:
    print()
    print("   (If a match was detected, the timing worked out — but it's not guaranteed!)")

## ✅ The Solution: Redis Lua Scripts (Atomic Operations)

The fix is to make the **check and write happen as a single atomic operation** — nothing can happen between them.

Redis supports **Lua scripts** that execute atomically. The entire script runs without interruption — no other Redis command can execute in the middle.

### Why is this possible? Because Redis is single-threaded.

Redis runs all commands on **one thread**. At any given moment only one command is being processed. A Lua script is treated like one "big command": once it starts, Redis won't run anything else until the whole script is done. That's a free atomicity guarantee — no locking, no transactions, no extra machinery.

> ⚠️ This also means: your Lua scripts must stay **fast**. If a script runs for 100ms, every other Redis client is blocked for 100ms. Keep them small (a few `GET`/`SET`/`HSET` calls, no loops over millions of items).

### The Key Design: User-Pair Keys

We need both directions of a swipe (A→B and B→A) to land on the **same Redis key**. We do this by always sorting the two user IDs:

```
User 6 swipes on User 16 → key = "swipe:6:16"
User 16 swipes on User 6 → key = "swipe:6:16"  (same key!)
```

The Redis hash for this key stores both users' swipe directions:
```
swipe:6:16 = {
    "6_swipe": "right",
    "16_swipe": "right"
}
```

> 🧩 **Scaling tip**: When Redis is sharded (Redis Cluster), keys are routed to shards by hashing the key name. Because we always compute the same key for the pair, *both swipes land on the same shard* — so the Lua script stays atomic even in a cluster.


In [ ]:
r = get_redis()

# The Lua script that atomically records a swipe and checks for match
SWIPE_SCRIPT = """
-- KEYS[1] = the user-pair key (e.g., "swipe:6:16")
-- ARGV[1] = field name for this swiper (e.g., "6_swipe")
-- ARGV[2] = swipe direction ("right" or "left")
-- ARGV[3] = field name for the other user (e.g., "16_swipe")

-- Step 1: Record this user's swipe
redis.call('HSET', KEYS[1], ARGV[1], ARGV[2])

-- Step 2: Set expiry so old swipes don't stick around forever (30 days)
redis.call('EXPIRE', KEYS[1], 2592000)

-- Step 3: Check the other user's swipe (atomic — no gap!)
local other_swipe = redis.call('HGET', KEYS[1], ARGV[3])

-- Step 4: Return the other user's swipe direction (or nil)
return other_swipe
"""

# Register the script with Redis for reuse
swipe_sha = r.script_load(SWIPE_SCRIPT)
print(f"✅ Lua script registered with SHA: {swipe_sha[:16]}...")
print()
print("💡 This script is ATOMIC — Redis executes it without interruption.")
print("   No other command can run between the HSET and the HGET.")
print("   This eliminates the race condition completely.")

In [ ]:
def get_swipe_key(user_a: int, user_b: int) -> str:
    """Create a consistent key for a pair of users.
    Always sorts IDs so both directions map to the same key."""
    smaller, larger = sorted([user_a, user_b])
    return f"swipe:{smaller}:{larger}"

def atomic_swipe(swiper_id: int, target_id: int, direction: str) -> bool:
    """Record a swipe and atomically check for match using Redis Lua script."""
    key = get_swipe_key(swiper_id, target_id)
    
    # Run the Lua script atomically
    other_swipe = r.evalsha(
        swipe_sha,
        1,       # number of KEYS
        key,     # KEYS[1]
        f"{swiper_id}_swipe",   # ARGV[1]: our field
        direction,               # ARGV[2]: our direction
        f"{target_id}_swipe"    # ARGV[3]: their field
    )
    
    # It's a match if both swiped right
    is_match = (direction == 'right' and other_swipe == 'right')
    return is_match

# Demonstrate the key pattern
print("🔑 User-pair key examples:")
print(f"   User 6 → User 16: key = '{get_swipe_key(6, 16)}'")
print(f"   User 16 → User 6: key = '{get_swipe_key(16, 6)}'  ← same key!")
print(f"   User 42 → User 7:  key = '{get_swipe_key(42, 7)}'")
print(f"   User 7 → User 42:  key = '{get_swipe_key(7, 42)}'  ← same key!")

In [ ]:
# Clean up previous test data
r.delete(get_swipe_key(6, 16))
execute("DELETE FROM swipes WHERE (swiper_id = 6 AND target_id = 16) OR (swiper_id = 16 AND target_id = 6)")

# Demonstrate atomic swipe: step by step
print("=" * 60)
print("Step-by-step atomic swipe demo: Luna(6) ↔ Oliver(16)")
print("=" * 60)

# Step 1: Oliver swipes right on Luna
match1 = atomic_swipe(16, 6, 'right')
print(f"\n1️⃣ Oliver(16) swipes RIGHT on Luna(6)")
print(f"   Match detected? {match1}")
print(f"   Redis key: {r.hgetall(get_swipe_key(6, 16))}")

# Step 2: Luna swipes right on Oliver
match2 = atomic_swipe(6, 16, 'right')
print(f"\n2️⃣ Luna(6) swipes RIGHT on Oliver(16)")
print(f"   Match detected? {match2}  🎉")
print(f"   Redis key: {r.hgetall(get_swipe_key(6, 16))}")

print()
if match2:
    print("✅ Match detected on the SECOND swipe — the Lua script atomically")
    print("   saw Oliver's existing 'right' swipe and returned it immediately.")

In [ ]:
# Now let's prove the race condition is FIXED with concurrent swipes

# Use users Ava(5) and Lucas(15) — test with fresh data
r.delete(get_swipe_key(5, 15))
execute("DELETE FROM swipes WHERE (swiper_id = 5 AND target_id = 15) OR (swiper_id = 15 AND target_id = 5)")

atomic_results = {}

def atomic_swipe_as(swiper, target, label):
    match = atomic_swipe(swiper, target, 'right')
    atomic_results[label] = match

print("🏁 Concurrent atomic swipes: Ava(5) ↔ Lucas(15)\n")

# Run 10 trials to show consistency
matches_detected = 0
for trial in range(10):
    r.delete(get_swipe_key(5, 15))
    atomic_results.clear()
    
    with ThreadPoolExecutor(max_workers=2) as pool:
        pool.submit(atomic_swipe_as, 5, 15, 'Ava→Lucas')
        pool.submit(atomic_swipe_as, 15, 5, 'Lucas→Ava')
    
    time.sleep(0.1)
    
    # At least one should detect the match
    if atomic_results.get('Ava→Lucas') or atomic_results.get('Lucas→Ava'):
        matches_detected += 1

print(f"✅ Matches detected: {matches_detected}/10 trials")
print()
if matches_detected == 10:
    print("🎉 PERFECT! Every concurrent swipe pair detected the match.")
    print("   The Redis Lua script guarantees atomicity — no more race conditions.")
else:
    print(f"   {matches_detected}/10 detected — Lua scripts guarantee at least one will.")

## 💾 Persisting to PostgreSQL

Redis gives us **speed and atomicity** for real-time match detection. But Redis is in-memory — if it crashes, we lose data. We need PostgreSQL as our **durable storage layer**.

The pattern is:
1. **Redis** handles the atomic swipe + match check (fast, consistent)
2. **PostgreSQL** stores the swipe and match permanently (durable)

If Redis loses data, the worst case is that a user has to swipe again. We never lose the permanent record in PostgreSQL.

In [ ]:
def handle_swipe(swiper_id: int, target_id: int, direction: str):
    """Complete swipe handler: Redis for atomic matching, PostgreSQL for durability."""
    
    # Step 1: Atomic match detection in Redis
    is_match = atomic_swipe(swiper_id, target_id, direction)
    
    # Step 2: Persist swipe to PostgreSQL
    execute(
        "INSERT INTO swipes (swiper_id, target_id, direction) VALUES (%s, %s, %s) ON CONFLICT DO NOTHING",
        (swiper_id, target_id, direction)
    )
    
    # Step 3: If match, persist the match too
    if is_match:
        # Always store with smaller ID first for consistency
        user1, user2 = sorted([swiper_id, target_id])
        execute(
            "INSERT INTO matches (user1_id, user2_id) VALUES (%s, %s) ON CONFLICT DO NOTHING",
            (user1, user2)
        )
        
        # Get names for display
        swiper_name = query("SELECT name FROM users WHERE id = %s", (swiper_id,))[0]['name']
        target_name = query("SELECT name FROM users WHERE id = %s", (target_id,))[0]['name']
        print(f"   🎉 MATCH! {swiper_name} and {target_name} like each other!")
    
    return is_match

print("📋 Complete swipe handler ready")
print("   1. Redis Lua script → atomic match detection")
print("   2. PostgreSQL INSERT → durable swipe record")
print("   3. PostgreSQL INSERT → durable match record (if match)")

In [ ]:
# Let's simulate a realistic swiping session for Daniel(19) and Chloe(9)
# Daniel already swiped right on Chloe (from seed data)
# Now Chloe opens the app and starts swiping...

# First, set up Daniel's existing swipe in Redis (simulating it was done earlier)
r.delete(get_swipe_key(9, 19))
atomic_swipe(19, 9, 'right')  # Daniel swiped right on Chloe earlier

print("📱 Chloe(9) opens the app and starts swiping...\n")

# Chloe swipes through her stack
print("Chloe swipes LEFT on Jack(20) — pass")
handle_swipe(9, 20, 'left')

print("Chloe swipes LEFT on Mason(17) — pass")
handle_swipe(9, 17, 'left')

print("Chloe swipes RIGHT on Daniel(19) — like!")
match = handle_swipe(9, 19, 'right')

print()
print("📊 Let's verify in PostgreSQL:")
matches = query("""
    SELECT m.id, u1.name as user1, u2.name as user2, m.matched_at
    FROM matches m
    JOIN users u1 ON m.user1_id = u1.id
    JOIN users u2 ON m.user2_id = u2.id
    ORDER BY m.matched_at DESC
    LIMIT 5
""")

print(f"\n{'Match ID':<10} {'User 1':<20} {'User 2':<20}")
print("-" * 50)
for m in matches:
    print(f"{m['id']:<10} {m['user1']:<20} {m['user2']:<20}")

## 📊 What's Inside Redis?

Let's inspect the Redis keys to see how swipe data is stored. Open **RedisInsight** at http://localhost:5540 to see this visually!

In [ ]:
# List all swipe keys in Redis
print("🔑 Redis swipe keys:\n")

keys = r.keys("swipe:*")
for key in sorted(keys):
    data = r.hgetall(key)
    ttl = r.ttl(key)
    
    print(f"  {key}")
    for field, value in data.items():
        emoji = "👍" if value == 'right' else "👎"
        print(f"    {emoji} {field} = {value}")
    print(f"    ⏳ TTL: {ttl // 86400} days")
    print()

print("💡 Each key is a user pair. Both swipe directions are in the same hash.")
print("   This is what makes atomic match detection possible.")

## ⚡ Performance: How Fast Is This?

At Tinder's scale (2B swipes/day = ~23,000 swipes/second), the swipe handler needs to be extremely fast. Let's benchmark our approach:

In [ ]:
import random

# Benchmark: How fast is the atomic swipe?
print("⏱️  Benchmarking atomic swipe (Redis Lua script)...\n")

times = []
for i in range(1000):
    user_a = random.randint(1, 1000)
    user_b = random.randint(1001, 2000)
    direction = random.choice(['right', 'left'])
    
    start = time.time()
    atomic_swipe(user_a, user_b, direction)
    elapsed = (time.time() - start) * 1000
    times.append(elapsed)

avg = sum(times) / len(times)
p95 = sorted(times)[int(len(times) * 0.95)]
p99 = sorted(times)[int(len(times) * 0.99)]

print(f"   Requests: 1,000 atomic swipes")
print(f"   Average:  {avg:.2f} ms")
print(f"   P95:      {p95:.2f} ms")
print(f"   P99:      {p99:.2f} ms")
print(f"   Max:      {max(times):.2f} ms")
print(f"   Throughput: {1000 / (sum(times)/1000):.0f} swipes/second (single-threaded)")
print()
print("💡 Redis Lua scripts execute in microseconds to low milliseconds.")
print("   At Tinder's scale, they'd shard across multiple Redis nodes.")
print("   The user-pair key pattern ensures both swipe directions hit the same shard.")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys from benchmarks
keys = r.keys("swipe:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")

# Clean up extra swipes/matches we created
execute("DELETE FROM matches WHERE id > 3")  # keep seed matches
execute("DELETE FROM swipes WHERE id > 13")   # keep seed swipes
print("🧹 Reset PostgreSQL to seed data")

## 📚 Summary

### Key Takeaways

1. **Naive check-then-write has a race condition** — concurrent swipes can both miss the match
2. **Redis Lua scripts are atomic because Redis is single-threaded** — one script = one command; nothing else runs until it finishes
3. **Keep Lua scripts short** — they block every other Redis client while they run
4. **User-pair keys** — sorting user IDs ensures both swipe directions map to the same Redis key (and the same shard in Redis Cluster)
5. **Hybrid storage** — Redis for real-time atomic matching, PostgreSQL for durable history
6. **Redis is fast** — sub-millisecond Lua script execution handles Tinder's ~23K swipes/second easily

### When NOT to use this pattern

- **If absolute durability of the match-detection step is required**, Redis alone isn't enough — combine with a durable write (PostgreSQL, as we did) or use a transactional DB approach (e.g., a stored procedure with `SERIALIZABLE` isolation).
- **If your operation is too complex for a short script**, consider a proper message queue (Kafka, RabbitMQ) with idempotent workers instead of Lua.

### The Architecture

```
User swipes → Redis Lua script (atomic check + write)
                 ├── Match? → Create match in PostgreSQL
                 │            → Send notification (Notebook 3)
                 └── No match → Save swipe to PostgreSQL
```

### Next Up

In **Notebook 3**, we'll build the **Real-Time Notification System** — using Redis Pub/Sub to instantly notify both users when a match occurs.
